In [1]:
#!pip install sentence_transformers
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

In [2]:
#!pip install "ag2[retrievechat]"

In [3]:
#!pip install pyautogen==0.7.2

Setting base for Retrive Chat

In [4]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance

from autogen.retrieve_utils import TEXT_FORMATS
config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


models to use:  ['gpt-35-turbo', 'gpt-4-vision-preview', 'dalle', 'gemini-pro', 'gemini-1.5-pro-001', 'gemini-1.5-pro', 'gemini-pro-vision']


In [5]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [6]:
config_list=[
    {
        "model":"gpt-6-luna",
        "api_key":"apikey",
        "api_type":"openai"
    }
]


In [7]:
import logging
logging.raiseExceptions = False

import agentops
agentops.init("f9de56a1-8b24-408e-a137-7e9638f47479")  
print("AgentOps initialized successfully")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor


AgentOps initialized successfully


🖇 AgentOps: You're on the agentops free plan 🤔


In [8]:
# 1. create an AssistantAgent instance named "assistant"
assistant=AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,  #Sets the maximum waiting time to 600 seconds.
        "cache_seed": 42,  #Enables reproducible response caching using seed 42.
        "config_list": config_list,
    },
)

In [9]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [10]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback¶

In [11]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)


Trying to create collection.


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-11-16 14:32:33,801 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 210 chunks.
2026-11-16 14:32:33,817 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Model gpt-6-luna not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['c024e487', 'e6980caf', '5597c053', '330fa96a', 'ae7d3b76', '7b45c013', '363de674', '5385d2ca', '0acb4ca9', 'dba71e1a', 'a3e8f8c3', 'ff103f25', 'a1dcc20c', '8b0b85f0', '797e9c8f', 'a0f75e72', 'bfa56a62', '687a8b1d', '94c25c16', '23d76c9a']]
Adding content of doc c024e487 to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc e6980caf to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


... (40 outputs left out)

[autogen.oai.client: 11-16 14:33:21] {573} WARNING - Model gpt-6-luna is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to ragproxyagent):

The excerpt argues that companies should own distinctive words or attributes in customers’ minds. IBM could brand powerful workstations “PMs” (“personal mainframes”); Lotus faced the challenge of growing beyond spreadsheets; and the Law of Attributes says to choose an attribute opposite to a competitor’s. White Castle illustrates sacrifice: focus on a limited offering at low prices.

The only revenue figure is that an average White Castle earns **more than $1 million annually**. The excerpt provides no exact figure, but its monthly equivalent is also more than $83,333:

```python
annual_revenue_threshold = 1_000_000
monthly_revenue_threshold = annual_revenue_threshold / 12

print(f"Average monthly revenue is more than ${monthly_revenue

🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


[autogen.oai.client: 11-16 14:33:24] {573} WARNING - Model gpt-6-luna is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to ragproxyagent):

The excerpt emphasizes owning a distinctive word or attribute in customers’ minds. IBM might call its workstations “PMs,” Lotus struggled to grow beyond spreadsheets, and the Law of Attributes recommends choosing a different attribute from competitors. White Castle illustrates focusing on a limited, low-cost offering.

The excerpt says an average White Castle earns **more than $1 million annually**:

```python
monthly_revenue = 1_000_000 / 12
print(f"More than ${monthly_revenue:,.2f} per month")
```

--------------------------------------------------------------------------------


🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [12]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

Model gpt-6-luna not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['9f39e066', '17300681', 'cea6b5ea', '7a179f8f', '5bb144e0', 'bbe2135b', '413c9d39', 'd48e9565', '7173c3da', '122e5901', '326cee40', 'b3f77776', 'de729254', '50525f55', 'd5c9a510', 'a1dcc20c', 'dba71e1a', 'f8240566', '3c4a80ea', '73eecb4d']]
Adding content of doc 9f39e066 to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc 17300681 to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc cea6b5ea to context.


... (25 outputs left out)

Adding content of doc a1dcc20c to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and assorted idea generators seem to
think that all their good ideas need is professional mar-
keting help.
Nothing could be further from the truth. Marketing
is a game fought in the min

🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [13]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

Model gpt-6-luna not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['a1dcc20c', '660c4d76', 'f8240566', '9f39e066', '7173c3da', 'a0f75e72', 'd15472f5', '73eecb4d', '7a179f8f', 'b0d70839', '1da5c235', 'bbe2135b', '413c9d39', 'be80b4fb', '4e6b17f7', '17300681', '69fa91d4', '55f04e11', '4b5df861', '7b45c013']]
Adding content of doc a1dcc20c to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc 660c4d76 to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc f8240566 to context.


... (29 outputs left out)

ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py

Context is: liant marketing program only to have one of the
immutable laws knock you flat if you don’t know what
they are.
Perhaps it’s human nature not to admit there are
things you can’t do. Certainly most marketers believe
that anything is achievable if you are energetic enough,
or creative enough, or determined enough. Especially
if you are willing to spend enough money.
Once you open your mind to the possibility that
there are laws of marketing, it’s easy to see what they
are. In truth, they are obv

🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit


🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [14]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

Model gpt-6-luna not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['73eecb4d', 'b0d70839', '660c4d76', '9f39e066', 'd15472f5', '7173c3da', '4b5df861', 'a1dcc20c', 'bbe2135b', '7a179f8f', 'a0f75e72', '1da5c235', 'f8240566', '7b45c013', '413c9d39', 'c944f7a7', '326cee40', '69fa91d4', '77ebd254', '3c4a80ea']]
Adding content of doc 73eecb4d to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc b0d70839 to context.


Model gpt-6-luna not found. Using cl100k_base encoding.


Adding content of doc 660c4d76 to context.


... (29 outputs left out)

ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. 

Context is: ing pad: Apple II, Commodore Pet, IMSAI 8080, MITS
Altair 8800, and Radio Shack TRS-80. Ask yourself,
which name is the simplest and easiest to remember?
17
THE LAW OF THE MIND
22 Laws of Marketing  10/31/02  12:23 PM  Page 174
The Law of 
Perception
Marketing is not a battle of products,
it’s a battle of perceptions.
18
22 Laws of Marketing  10/31/02  12:23 PM  Page 18Many people think marketing is a battle of prod-
ucts. In the long run, they figure, the best product will
win.
Marketing people are preoccupied with doing
research and “getting

🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit


🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'
